# Phase 0 + 1 — Setup, EDA, Scoring Harness, Validation Split

Follows `IMPLEMENTATION_PLAN.md`. Run top to bottom in the **`ura`** conda env.

1. Environment check (Python, GPU/torch)
2. Load data + EDA stats
3. Scoring harness unit tests (exact competition metric)
4. Grouped train/val split (no thumbnail leakage)

In [1]:
import sys, os
from pathlib import Path

# Make src/ importable regardless of where the kernel started
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    pass
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('src on path  =', SRC.exists())

import platform
print('Python       =', platform.python_version())
try:
    import torch
    print('torch        =', torch.__version__, '| CUDA available =', torch.cuda.is_available(),
          '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'))
except Exception as e:
    print('torch        = not importable yet:', e)

PROJECT_ROOT = c:\GET_A_JOB\Vin Smart Future Intern\URA_Hackathon_2026\URA_Hackathon_Kaggle_Competition_2026
src on path  = True
Python       = 3.13.12
torch        = 2.11.0+cpu | CUDA available = False | CPU only


In [2]:
import pandas as pd
import config, data, scoring
from importlib import reload
reload(config); reload(data); reload(scoring)

labels = data.load_train_labels()
test_ids = data.load_test_ids()
print(f'train labels : {len(labels):,} rows')
print(f'test ids     : {len(test_ids):,} rows')
labels.head()

train labels : 4,892 rows
test ids     : 2,006 rows


,image_id,ocr_text,product_name
0,img_0001,,
1,img_0002,,
2,img_0003,Hệ Lụy Đồ Hộp,
3,img_0004,,
4,img_0005,MUKBANG Review ĐỒ HỘP HẠ LONG,Đồ Hộp Hạ Long


## EDA — empty rates, product distribution, OCR length

In [3]:
empty_ocr = (labels['ocr_text'] == '').mean()
empty_prod = (labels['product_name'] == '').mean()
both_empty = ((labels['ocr_text'] == '') & (labels['product_name'] == '')).mean()
print(f'empty ocr_text     : {empty_ocr:.1%}')
print(f'empty product_name : {empty_prod:.1%}')
print(f'both empty         : {both_empty:.1%}  (these score full credit if we predict empty)')
print(f'unique products    : {labels.loc[labels.product_name!="", "product_name"].nunique()}')

import numpy as np
ol = labels.loc[labels.ocr_text != '', 'ocr_text'].str.len()
print(f'\nOCR length  median={int(ol.median())}  mean={ol.mean():.0f}  p90={int(ol.quantile(0.9))}  max={ol.max()}')
pl = labels.loc[labels.product_name != '', 'product_name'].str.split().apply(len)
print(f'Product tokens  median={int(pl.median())}  mean={pl.mean():.2f}  max={pl.max()}')

empty ocr_text     : 20.2%
empty product_name : 40.7%
both empty         : 19.6%  (these score full credit if we predict empty)
unique products    : 495

OCR length  median=100  mean=124  p90=261  max=500
Product tokens  median=4  mean=3.73  max=36


In [4]:
# Top product strings (raw, not yet normalized)
top = labels.loc[labels.product_name != '', 'product_name'].value_counts().head(25)
top.to_frame('count')

,count
product_name,
ĐỒ HỘP HẠ LONG,302
Đồ hộp Hạ Long,275
đồ hộp Hạ Long,89
patê CỘT ĐÈN HẢI PHÒNG,63
Pate Cột Đèn Hải Phòng,54
NAN,52
sữa NAN,52
pate cột đèn,45
Nestlé NAN,43


In [5]:
# Brand-token frequency: lowercased token counts across product_name — seeds the gazetteer
from collections import Counter
c = Counter()
for p in labels.loc[labels.product_name != '', 'product_name']:
    c.update(p.lower().split())
brand_tokens = pd.Series(dict(c.most_common(40)), name='count')
brand_tokens.to_frame()

,count
long,880
hộp,853
hạ,853
đồ,836
nan,615
cột,483
đèn,476
nestlé,440
sữa,355
hải,353


## Scoring harness — unit tests (exact competition metric)

In [6]:
# token_f1 cases\nassert scoring.token_f1('Vinamilk Flex', 'vinamilk flex') == 1.0          # case-insensitive\nassert scoring.token_f1('', '') == 1.0                                     # both empty -> perfect\nassert scoring.token_f1('Vinamilk', '') == 0.0                             # one empty -> 0\nassert scoring.token_f1('', 'Vinamilk') == 0.0\n# partial overlap: pred 'Ha Long Canfoco' vs gt 'Ha Long Canfoco Pate' -> P=3/3, R=3/4 -> F1=0.857\nassert abs(scoring.token_f1('Ha Long Canfoco Pate', 'Ha Long Canfoco') - 0.8571) < 1e-3\nprint('partial F1:', round(scoring.token_f1('Ha Long Canfoco Pate', 'Ha Long Canfoco'), 4))\n\n# cer cases\nassert scoring.cer('', '') == 0.0\nassert scoring.cer('', 'abc') == 1.0\nassert scoring.cer('abc', 'abc') == 0.0\nassert abs(scoring.cer('abcd', 'abxd') - 0.25) < 1e-9                       # 1 sub / 4\nprint('cer abcd vs abxd:', scoring.cer('abcd', 'abxd'))\nprint('token_f1 / cer unit tests passed')

In [7]:
# Sanity: score an all-empty submission against train labels (informative, not the 0.25 ref which is on test)
empty_sub = labels[['image_id']].copy()
empty_sub['ocr_text'] = ''
empty_sub['product_name'] = ''
print('all-empty vs train labels:', scoring.composite_score(labels, empty_sub, return_components=True))

# Oracle: copy GT exactly -> must be 1.0
print('oracle (GT==pred)      :', scoring.composite_score(labels, labels, return_components=True))

all-empty vs train labels: {'composite': 0.3251, 'product_f1': 0.4074, 'ocr_term': 0.2018, 'avg_cer': 0.7982}
oracle (GT==pred)      : {'composite': 1.0, 'product_f1': 1.0, 'ocr_term': 1.0, 'avg_cer': 0.0}


## Validation split — grouped by OCR text (no thumbnail leakage)

In [8]:
split = data.make_split(labels)
n_val = (split.split == 'val').sum()
print(f"train: {(split.split=='train').sum():,}  |  val: {n_val:,}  ({n_val/len(split):.1%})")

# Leakage check: no OCR-text group spans both sides
g = split.groupby('group')['split'].nunique()
print('groups spanning both splits (must be 0):', int((g > 1).sum()))

# Empty balance across splits
for s in ['train', 'val']:
    sub = split[split.split == s]
    print(f"  {s}: empty_ocr={ (sub.ocr_text=='').mean():.1%}  empty_prod={ (sub.product_name=='').mean():.1%}")

data.save_split(split)
print('\nsaved splits/ -> train_ids.txt, val_ids.txt')

train: 3,912  |  val: 980  (20.0%)
groups spanning both splits (must be 0): 0
  train: empty_ocr=25.2%  empty_prod=44.5%
  val: empty_ocr=0.0%  empty_prod=25.8%

saved splits/ -> train_ids.txt, val_ids.txt


In [9]:
# Baseline-on-val reference: rules-only product extractor scored on GT ocr_text (oracle OCR)
# This isolates the product extractor ceiling before real OCR is plugged in (Phase 2).
val = split[split.split == 'val'].copy()
val_pred = val[['image_id', 'ocr_text']].copy()
val_pred['product_name'] = ''   # placeholder: empty product baseline
print('val, empty-product baseline:', scoring.composite_score(val[['image_id','ocr_text','product_name']], val_pred, return_components=True))
print('\nPhase 0/1 complete. Next: Phase 2 OCR bake-off (src/ocr_engines.py + run_ocr.py).')

val, empty-product baseline: {'composite': 0.5549, 'product_f1': 0.2582, 'ocr_term': 1.0, 'avg_cer': 0.0}

Phase 0/1 complete. Next: Phase 2 OCR bake-off (src/ocr_engines.py + run_ocr.py).
